### Load dataset

In [1]:
import pandas as pd

df = pd.read_csv("data/cars.csv")
df.head()

,brand,model,year,mileage,fuel,transmission,price
0,Toyota,Corolla,2019,60000,Petrol,Automatic,18500
1,Honda,Civic,2018,80000,Petrol,Manual,15000
2,Toyota,Camry,2020,40000,Petrol,Automatic,24000
3,Ford,Focus,2017,90000,Diesel,Manual,12000
4,BMW,3 Series,2019,50000,Petrol,Automatic,32000


### Preparing for LLM

In [5]:
# Create text + metadata manually
documents = [
    {"id": str(i),
     "text": f"{row['brand']} {row['model']} {row['year']} {row['mileage']}km {row['fuel']} {row['transmission']}",
     "metadata": {"price": row['price']}}
    for i, row in df.iterrows()
]

### Build Embedding

In [6]:
from sentence_transformers import SentenceTransformer

# Load embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Compute embeddings for all documents
embeddings = [embedding_model.encode(doc["text"]) for doc in documents]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### Build Chroma V-Db

In [7]:
import chromadb

# Initialize Chroma client
client = chromadb.Client()

# Create collection
collection = client.create_collection("car_prices")

# Add documents + embeddings
for doc, emb in zip(documents, embeddings):
    collection.add(
        ids=[doc["id"]],
        documents=[doc["text"]],
        metadatas=[doc["metadata"]],
        embeddings=[emb.tolist()]  # must convert to list
    )

print("✅ Chroma index built successfully!")

✅ Chroma index built successfully!


## RAG Prediction

In [8]:
def predict_price(description, k=3):
    # Compute embedding for query
    query_emb = embedding_model.encode(description).tolist()
    
    # Retrieve top k similar documents
    results = collection.query(query_embeddings=[query_emb], n_results=k)
    
    # Average the prices of top-k hits
    prices = [r['price'] for r in results['metadatas'][0]]
    avg_price = sum(prices) / len(prices) if prices else 0
    
    return round(avg_price, 2)

In [9]:
query = "Toyota Corolla 2019 60000 km petrol automatic"
predicted_price = predict_price(query)
print(f"Estimated resale price: ${predicted_price}")

Estimated resale price: $24833.33


In [10]:
import gradio as gr

def gradio_predict(desc):
    price = predict_price(desc)
    return f"Estimated resale price: ${price}"

iface = gr.Interface(
    fn=gradio_predict,
    inputs="text",
    outputs="text",
    title="Car Resale Price Predictor"
)

iface.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
